# Sound in Novels: Henry James and Marcel Proust
## Chapter-wise sound density, normalized length, loudness variation and music

**Texts**
* Henry James, *What Maisie Knew* (1897)
* Henry James, *The Golden Bowl* (1904, complete)
* Marcel Proust, *Swann in Love* (1913; English translation)

**What this notebook computes**
1. **Normalized length comparison**: word counts, chapter counts and chapter lengths, plus every position in a text expressed as a percentage of narrative time (0–100 %), so that books of very different lengths can be overlaid.
2. **Chapter-wise sound event density (SED)** from the chapter annotations (`<chapter counter="…"/>` milestones). *Swann in Love* has no chapter milestones, so it is cut into paragraph-respecting segments of about 3,000 words (close to the average James chapter length).
3. **Loudness level statistics**: mean and **standard deviation** of the loudness labels (1 = very quiet … 5 = very loud) per text, per sound type (character / ambient), per chapter, and as a rolling curve over narrative time.
4. **Music**: where and how often music is represented, how much of it the sound-event annotation actually captures, and how loud it is labeled.

All charts are interactive Plotly figures. They are written as standalone HTML files into `html_visualizations/`, and the last cell assembles one HTML report for the class.

The approach follows `Sound_in_Dickens.ipynb` (word tokenizer `\b\w+\b`, `np.std` with ddof = 0, character vs. ambient sound), with two changes noted where they occur:
* sound events are kept in **lists, not dictionaries keyed by span**, so a span that recurs (e.g. *"she said"*) is counted every time it occurs;
* the loudness labels are read straight from the XML, and the CSV from the labeling notebook is only used as a cross-check.

In [1]:
#!pip install pandas numpy lxml plotly scipy
import re, ast, html, json
from pathlib import Path

import numpy as np
import pandas as pd
from lxml import etree
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from scipy.stats import levene, kruskal, ttest_ind

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)

## Setup for Google Colab
If you opened this notebook in **Google Colab**, the next cell gets the data:
* it first tries to download it from the GitHub repository named in `REPO`;
* if that is not possible (for example because the repository is private), an **upload button** appears. Click it and choose the repository ZIP file you received from your instructor, or the three `.xml` files.

Anywhere else (Jupyter on your computer, Binder) this cell does nothing.

In [2]:
import sys, subprocess, os, zipfile, shutil, tempfile
REPO = "SvenjaGuhr/Sound_and_Loudness_in_Novels_by_James_and_Proust"          # ← the GitHub address of this repository, e.g. "jdoe/sound-in-james-proust"

if "google.colab" in sys.modules and not Path("data").exists():
    r = subprocess.run(["git", "clone", "-q", "--depth", "1", f"https://github.com/{REPO}.git", "repo"],
                       capture_output=True, env={**os.environ, "GIT_TERMINAL_PROMPT": "0"})
    if r.returncode == 0:
        os.chdir("repo")
        print("Data downloaded from GitHub.")
    else:
        print("The data could not be downloaded from GitHub.\n"
              "Click 'Choose files' below and select the repository ZIP file (or the three .xml files).")
        from google.colab import files
        uploaded = files.upload()
        Path("data").mkdir(exist_ok=True)
        for name in uploaded:
            if name.lower().endswith(".zip"):
                tmp = Path(tempfile.mkdtemp())
                with zipfile.ZipFile(name) as z:
                    z.extractall(tmp)
                for f in list(tmp.rglob("*.xml")) + list(tmp.rglob("English_texts_LL_predicted.csv")):
                    shutil.copy(f, Path("data") / f.name)
            elif name.lower().endswith((".xml", ".csv")):
                shutil.move(name, Path("data") / name)
        print("Files in data/:", sorted(p.name for p in Path("data").iterdir()))

Data downloaded from GitHub.


## Configuration
The XML files live in the folder `data/`. The settings below are the knobs you can turn: change a number, then choose **Run → Run all cells** (Jupyter) or **Runtime → Run all** (Colab).

In [3]:
DATA_DIR = Path("data") if Path("data").exists() else Path(".")   # folder with the XML files
CSV_PATH = DATA_DIR / "English_texts_LL_predicted.csv"
OUT_DIR = Path("html_visualizations")      # standalone HTML charts go here
OUT_DIR.mkdir(exist_ok=True)
REPORT_PATH = Path("Sound_in_James_and_Proust_report.html")

PROUST_SEGMENT_WORDS = 3000   # size of pseudo-chapters for Swann in Love
N_BINS = 20                   # bins of normalized narrative time (5 % each)
ROLLING_EVENTS = 60           # window (in sound events) for rolling loudness mean / std
MUSIC_WINDOW = 25             # tokens around an unambiguous music word that make ambiguous words count as music

TEXTS = {
    "What Maisie Knew": dict(file="James_Henry_What_Maisie_Knew.xml", author="Henry James", year=1897, color="#2a78d6"),
    "The Golden Bowl":  dict(file="James_Henry_The_Golden_Bowl.xml",  author="Henry James", year=1904, color="#eb6834"),
    "Swann in Love":    dict(file="Proust_Swann_in_Love.xml",          author="Marcel Proust", year=1913, color="#1baf7a"),
}
ORDER = list(TEXTS)
TEXT_COLORS = {t: v["color"] for t, v in TEXTS.items()}
TYPE_COLORS = {"character": "#4a3aa7", "ambient": "#eda100"}
MUSIC_COLORS = {"strict": "#d55181", "contextual": "#e87ba4", "figurative": "#9e9d97"}

## Parsing the TEI files
The chapters are **milestones** (`<chapter counter="3">III</chapter>`), not containers, so the parser walks the body in reading order and assigns every word to the chapter most recently opened. Heading text (chapter numerals, *BOOK FIRST*, *PART SECOND*, the title line) is not counted as words.

The walk produces two tables:
* `tokens`: one row per word, with chapter, paragraph break flag, and the id of the sound event the word belongs to (−1 if none);
* `events`: one row per `<character_sound>` / `<ambient_sound>` element, with its loudness label (NaN if unlabeled).

In [4]:
NS = {"tei": "http://www.tei-c.org/ns/1.0"}
TOKEN_RE = re.compile(r"\b\w+\b")
PARA_RE = re.compile(r"\n\s*\n")
SOUND_TAGS = {"character_sound": "character", "ambient_sound": "ambient"}
HEADING_TAGS = {"chapter", "book", "part"}


def to_float(v):
    try:
        return float(v)
    except (TypeError, ValueError):
        return np.nan            # missing label or 'S'


def parse_tei(path, text_name, fixes=()):
    raw = Path(path).read_text(encoding="utf-8")
    for old, new in fixes:                       # documented annotation repairs, see below
        assert raw.count(old) == 1, f"fix not found exactly once in {path}: {old[:60]}"
        raw = raw.replace(old, new)
    root = etree.fromstring(raw.lstrip().encode("utf-8"))
    tree = root.getroottree()
    body = tree.find(".//tei:body", NS)
    title = tree.findtext(".//tei:title", namespaces=NS) or ""

    words, starts, ends, chap, chap_label, book, part, ev_ids, para = [], [], [], [], [], [], [], [], []
    plain = []                    # running plain text, for keyword-in-context snippets
    offset = [0]
    events = []
    st = dict(chapter=None, chapter_label=None, book=None, part=None, para_break=True)

    def add_text(s, ev):
        if not s:
            return
        base = offset[0]
        pieces = PARA_RE.split(s)
        # character offsets of the pieces inside s
        pos, piece_offsets = 0, []
        for p in pieces:
            i = s.find(p, pos)
            piece_offsets.append(i)
            pos = i + len(p)
        for k, (p, po) in enumerate(zip(pieces, piece_offsets)):
            if k > 0:
                st["para_break"] = True
            for m in TOKEN_RE.finditer(p):
                words.append(m.group())
                starts.append(base + po + m.start()); ends.append(base + po + m.end())
                chap.append(st["chapter"]); chap_label.append(st["chapter_label"])
                book.append(st["book"]); part.append(st["part"])
                ev_ids.append(ev); para.append(st["para_break"])
                st["para_break"] = False
        plain.append(s)
        offset[0] += len(s)

    def walk(el, ev):
        if not isinstance(el.tag, str):        # comments / processing instructions
            return
        tag = etree.QName(el).localname
        if tag in HEADING_TAGS:
            label = (el.text or "").strip()
            if tag == "chapter":
                st["chapter"] = int(el.get("counter"))
                st["chapter_label"] = label or ("Prologue" if el.get("counter") == "0" else el.get("counter"))
            else:
                st[tag] = label
            st["para_break"] = True
            plain.append("\n\n"); offset[0] += 2
            return
        if tag in SOUND_TAGS:
            eid = len(events)
            events.append(dict(event_id=eid, type=SOUND_TAGS[tag],
                               loudness=to_float(el.get("loudness")),
                               loudness_raw=el.get("loudness"),
                               span=" ".join("".join(el.itertext()).split()),
                               start=len(words)))
            add_text(el.text, eid)
            for ch in el:
                walk(ch, eid)
                add_text(ch.tail, eid)
            events[eid]["end"] = len(words)
            return
        add_text(el.text, ev)
        for ch in el:
            walk(ch, ev)
            add_text(ch.tail, ev)

    # the body starts with the title line ("The Golden Bowl\n\n…"): drop it
    first = body.text or ""
    stripped = first.lstrip()
    if title and stripped.startswith(title):
        body.text = stripped[len(title):]
    walk(body, -1)

    tok = pd.DataFrame(dict(word=words, char_start=starts, char_end=ends, chapter=chap,
                            chapter_label=chap_label, book=book, part=part,
                            event_id=ev_ids, para_break=para))
    tok.insert(0, "text", text_name)
    tok["i"] = np.arange(len(tok))
    tok["rel_pos"] = tok["i"] / len(tok) * 100
    ev = pd.DataFrame(events)
    ev.insert(0, "text", text_name)
    ev["n_tokens"] = ev["end"] - ev["start"]
    return tok, ev, "".join(plain)

### Annotation repair in *What Maisie Knew*
The Maisie file is well-formed XML but contains two annotation errors that cancel each other out: a stray opening tag in chapter XX (*Susan `<character_sound>`Ash, in the hall …*) and a stray closing tag in chapter XXVI (*… with Mrs. Beale`</character_sound>`'s loud cry at her`</character_sound>`*). Together they turn 26,000 words (chapters XX–XXVI) into one giant "sound event", which would inflate every density measure for Maisie. The fixes below are applied in memory before parsing (the file on disk is not changed); the sanity check after parsing lists the longest events so that similar problems become visible.

In [5]:
FIXES = {
    "What Maisie Knew": [
        ("Susan <character_sound>Ash, in the hall, ", "Susan Ash, in the hall, "),
        ("<character_sound>it leaped out with Mrs. Beale</character_sound>'s loud cry at her</character_sound>",
         "<character_sound>it leaped out with Mrs. Beale's loud cry at her</character_sound>"),
    ],
}

tokens_by_text, events_by_text, plain_by_text = {}, {}, {}
for name, meta in TEXTS.items():
    tok, ev, plain = parse_tei(DATA_DIR / meta["file"], name, FIXES.get(name, ()))
    tokens_by_text[name], events_by_text[name], plain_by_text[name] = tok, ev, plain
    print(f"{name:18s} words: {len(tok):>7,}   sound events: {len(ev):>5,}   "
          f"chapters: {tok['chapter'].nunique()}")

What Maisie Knew   words:  98,361   sound events: 1,662   chapters: 32
The Golden Bowl    words: 212,368   sound events: 2,011   chapters: 42
Swann in Love      words:  91,562   sound events:   945   chapters: 0


In [6]:
# Sanity check: the longest sound events per text (an event of hundreds of words signals a tagging error)
for name, ev in events_by_text.items():
    top = ev.nlargest(3, "n_tokens")
    print(f"{name}: longest events = {top['n_tokens'].tolist()} words")
    print("   ", top["span"].iloc[0][:120])

What Maisie Knew: longest events = [20, 19, 18] words
    Maisie heard afresh among the pleasant sounds of the closing day that steel click of Ida's change of mind
The Golden Bowl: longest events = [28, 28, 27] words
    There was really nothing they had talked of together with more intimateand familiar pleasantry than of the license and p
Swann in Love: longest events = [29, 16, 15] words
    earlier, the sound of the door-bell, swelling in the depths of his abyss of sleep into the clangour of an alarum, had en


### Units of analysis: chapters, and segments for Proust
James's chapters come from the annotation (*What Maisie Knew*: the unnumbered opening section + chapters I–XXXI; *The Golden Bowl*: chapters I–XLII in two books and six parts). *Swann in Love* has no chapter milestones; it is divided at the first paragraph break after every ~3,000 words. A last segment shorter than a third of that is merged into the one before it.

In [7]:
def paragraph_segments(tok, target):
    seg, count, out = 1, 0, []
    for pb in tok["para_break"].to_numpy():
        if pb and count >= target:
            seg += 1
            count = 0
        out.append(seg)
        count += 1
    out = np.array(out)
    last = out == out.max()
    if last.sum() < target / 3 and out.max() > 1:
        out[last] = out.max() - 1
    return out


for name, tok in tokens_by_text.items():
    if tok["chapter"].isna().all():
        tok["unit"] = paragraph_segments(tok, PROUST_SEGMENT_WORDS)
        tok["unit_label"] = "§" + tok["unit"].astype(str)
        tok["unit_kind"] = "segment"
    else:
        tok["unit"] = tok["chapter"].astype(int)
        tok["unit_label"] = tok["chapter_label"]
        tok["unit_kind"] = "chapter"

    ev = events_by_text[name]
    ev["unit"] = tok["unit"].to_numpy()[ev["start"].clip(upper=len(tok) - 1)]
    ev["unit_label"] = tok["unit_label"].to_numpy()[ev["start"].clip(upper=len(tok) - 1)]
    ev["rel_pos"] = ev["start"] / len(tok) * 100

tokens = pd.concat(tokens_by_text.values(), ignore_index=True)
events = pd.concat(events_by_text.values(), ignore_index=True)
events.head()

,text,event_id,type,loudness,loudness_raw,span,start,end,n_tokens,unit,unit_label,rel_pos
0,What Maisie Knew,0,character,NaN,None,whom she proposed,552,555,3,0,Prologue,0.561198
1,What Maisie Knew,1,character,3.0,3.0,Ida cried,612,614,2,0,Prologue,0.622198
2,What Maisie Knew,2,character,NaN,None,"The good lady, for a moment, made no reply",683,692,9,0,Prologue,0.694381
3,What Maisie Knew,3,character,NaN,None,her silence was a grim judgement of the whole ...,692,704,12,0,Prologue,0.703531
4,What Maisie Knew,4,character,3.0,3.0,she at last exclaimed,707,711,4,0,Prologue,0.718781


### Cross-check with the CSV from the loudness-labeling notebook
The CSV lists the extracted spans per file. Its counts should equal the number of elements found in the XML. (Maisie has one character sound fewer here because of the repair above.)

In [8]:
if CSV_PATH.exists():
    csv_df = pd.read_csv(CSV_PATH)
    rows = []
    for name, meta in TEXTS.items():
        r = csv_df.loc[csv_df["File"] == meta["file"]]
        if r.empty:
            continue
        r = r.iloc[0]
        ev = events_by_text[name]
        rows.append(dict(text=name,
                         csv_ambient=len(ast.literal_eval(r["Ambient Sound Spans"])),
                         xml_ambient=(ev["type"] == "ambient").sum(),
                         csv_character=len(ast.literal_eval(r["Character Sound Spans"])),
                         xml_character=(ev["type"] == "character").sum()))
    display(pd.DataFrame(rows))
else:
    print("CSV not found, skipping cross-check")

,text,csv_ambient,xml_ambient,csv_character,xml_character
0,What Maisie Knew,168,168,1495,1494
1,The Golden Bowl,174,174,1837,1837
2,Swann in Love,116,116,829,829


## 1. Normalized length comparison
*The Golden Bowl* is more than twice as long as the other two texts. Three normalizations make the books comparable:
* **relative length**: words divided by the longest text (*The Golden Bowl* = 1.00);
* **rates per 1,000 words**, used for every density measure below;
* **narrative time**: each word's position as a percentage of the whole text, so that the first 5 % of *Maisie* can be set next to the first 5 % of *The Golden Bowl*.

In [9]:
unit_words = (tokens.groupby(["text", "unit", "unit_label", "unit_kind"], sort=False)
              .size().rename("words").reset_index())

length_rows = []
for name in ORDER:
    tok = tokens_by_text[name]
    uw = unit_words[unit_words.text == name]["words"]
    length_rows.append(dict(
        text=name, author=TEXTS[name]["author"], year=TEXTS[name]["year"],
        words=len(tok),
        units=len(uw), unit_kind=tokens_by_text[name]["unit_kind"].iat[0],
        mean_unit_words=round(uw.mean()), median_unit_words=round(uw.median()),
        min_unit_words=uw.min(), max_unit_words=uw.max(),
        unit_length_cv=round(uw.std(ddof=0) / uw.mean(), 2),
    ))
length_df = pd.DataFrame(length_rows)
length_df["relative_length"] = (length_df["words"] / length_df["words"].max()).round(2)
length_df["relative_to_Maisie"] = (length_df["words"] / length_df.loc[0, "words"]).round(2)
length_df

,text,author,year,words,units,unit_kind,mean_unit_words,median_unit_words,min_unit_words,max_unit_words,unit_length_cv,relative_length,relative_to_Maisie
0,What Maisie Knew,Henry James,1897,98361,32,chapter,3074,3028,1093,6310,0.38,0.46,1.00
1,The Golden Bowl,Henry James,1904,212368,42,chapter,5056,4811,2833,7382,0.24,1.00,2.16
2,Swann in Love,Marcel Proust,1913,91562,27,segment,3391,3162,3003,5049,0.16,0.43,0.93


## 2. Sound event density (SED) per chapter
Two density measures, both normalized by length:
* **events per 1,000 words**: number of sound events that start in the chapter ÷ words × 1,000;
* **SED (% of words)**: share of the chapter's words that lie inside a sound-event span. This is the quantity behind `SED_without_nan` in `Sound_in_Dickens.ipynb` (sound events × average event length ÷ words × 100), computed here directly from the tokens.

The notebook also stores how many events in each chapter carry a loudness label.

In [10]:
def rate(n, words, per=1000):
    return np.where(words > 0, n / words * per, np.nan)


ev_counts = (events.groupby(["text", "unit", "type"]).size().unstack(fill_value=0)
             .reindex(columns=["character", "ambient"], fill_value=0).reset_index())
in_sound = (tokens[tokens.event_id >= 0].groupby(["text", "unit"]).size()
            .rename("words_in_sound").reset_index())

chap = (unit_words.merge(ev_counts, on=["text", "unit"], how="left")
        .merge(in_sound, on=["text", "unit"], how="left").fillna({"character": 0, "ambient": 0, "words_in_sound": 0}))
chap["events"] = chap["character"] + chap["ambient"]
chap["events_per_1k"] = rate(chap["events"], chap["words"])
chap["character_per_1k"] = rate(chap["character"], chap["words"])
chap["ambient_per_1k"] = rate(chap["ambient"], chap["words"])
chap["SED_pct"] = chap["words_in_sound"] / chap["words"] * 100
chap["ambient_share"] = chap["ambient"] / chap["events"].replace(0, np.nan)
first_pos = tokens.groupby(["text", "unit"])["rel_pos"].min().rename("starts_at_pct").reset_index()
chap = chap.merge(first_pos, on=["text", "unit"])

# loudness per chapter (labelled events only), population std as in the Dickens notebook
lab = events.dropna(subset=["loudness"])
loud_chap = lab.groupby(["text", "unit"])["loudness"].agg(
    loud_mean="mean", loud_std=lambda s: np.std(s, ddof=0), loud_n="size").reset_index()
chap = chap.merge(loud_chap, on=["text", "unit"], how="left")
chap["text"] = pd.Categorical(chap["text"], ORDER, ordered=True)
chap = chap.sort_values(["text", "unit"]).reset_index(drop=True)
chap.round(2).head(12)

,text,unit,unit_label,unit_kind,words,character,ambient,words_in_sound,events,events_per_1k,character_per_1k,ambient_per_1k,SED_pct,ambient_share,starts_at_pct,loud_mean,loud_std,loud_n
0,What Maisie Knew,0,Prologue,chapter,1667,8,2,51,10,6.00,4.80,1.20,3.06,0.20,0.00,3.00,0.00,4
1,What Maisie Knew,1,I,chapter,1416,19,2,87,21,14.83,13.42,1.41,6.14,0.10,1.69,3.20,0.60,10
2,What Maisie Knew,2,II,chapter,1396,14,3,100,17,12.18,10.03,2.15,7.16,0.18,3.13,3.08,0.27,13
3,What Maisie Knew,3,III,chapter,1093,16,0,89,16,14.64,14.64,0.00,8.14,0.00,4.55,3.00,0.00,12
4,What Maisie Knew,4,IV,chapter,1512,15,5,75,20,13.23,9.92,3.31,4.96,0.25,5.66,3.19,0.35,8
5,What Maisie Knew,5,V,chapter,1721,18,9,116,27,15.69,10.46,5.23,6.74,0.33,7.20,3.00,0.37,11
6,What Maisie Knew,6,VI,chapter,2146,33,2,158,35,16.31,15.38,0.93,7.36,0.06,8.95,3.18,0.37,19
7,What Maisie Knew,7,VII,chapter,2514,55,4,262,59,23.47,21.88,1.59,10.42,0.07,11.13,3.11,0.52,33
8,What Maisie Knew,8,VIII,chapter,3676,62,10,322,72,19.59,16.87,2.72,8.76,0.14,13.69,3.17,0.53,53
9,What Maisie Knew,9,IX,chapter,3019,34,4,170,38,12.59,11.26,1.32,5.63,0.11,17.43,2.86,0.64,22


In [11]:
# Text-level summary
summary_rows = []
for name in ORDER:
    tok, ev = tokens_by_text[name], events_by_text[name]
    w = len(tok)
    lab_ev = ev.dropna(subset=["loudness"])
    c, a = lab_ev[lab_ev.type == "character"]["loudness"], lab_ev[lab_ev.type == "ambient"]["loudness"]
    cu = chap[chap.text == name]
    summary_rows.append(dict(
        text=name, words=w,
        sound_events=len(ev), character_events=(ev.type == "character").sum(), ambient_events=(ev.type == "ambient").sum(),
        events_per_1k=len(ev) / w * 1000,
        character_per_1k=(ev.type == "character").sum() / w * 1000,
        ambient_per_1k=(ev.type == "ambient").sum() / w * 1000,
        SED_pct=(tok.event_id >= 0).mean() * 100,
        mean_event_length_words=ev["n_tokens"].mean(),
        labelled_share=ev["loudness"].notna().mean(),
        chapter_SED_std=cu["events_per_1k"].std(ddof=0),       # how unevenly sound is spread over chapters
        chapter_SED_min=cu["events_per_1k"].min(), chapter_SED_max=cu["events_per_1k"].max(),
    ))
summary = pd.DataFrame(summary_rows)
summary.round(3)

,text,words,sound_events,character_events,ambient_events,events_per_1k,character_per_1k,ambient_per_1k,SED_pct,mean_event_length_words,labelled_share,chapter_SED_std,chapter_SED_min,chapter_SED_max
0,What Maisie Knew,98361,1662,1494,168,16.897,15.189,1.708,7.096,4.200,0.615,5.136,5.999,26.443
1,The Golden Bowl,212368,2011,1837,174,9.469,8.650,0.819,4.276,4.516,0.658,4.278,2.907,19.226
2,Swann in Love,91562,945,829,116,10.321,9.054,1.267,4.138,4.010,0.631,5.112,4.318,23.099


## 3. Loudness: mean and standard deviation
Loudness labels come from the dictionary approach (`20250921_loudness_level_labeling…ipynb`): each sound event receives the average loudness of the matched sound words, on a scale from 1 (very quiet) to 5 (very loud). Unlabeled events are excluded.

The standard deviation is the population standard deviation (`np.std`, ddof = 0) used in the Dickens notebook. It answers the question: *how varied is the acoustic range of the text?* A small value means most sounds sit at the same level (typically speech, 3.0), a larger one means the text moves between whispers and noise.

`text_loudness_average` reproduces the Dickens notebook's definition (the unweighted mean of the character and ambient averages); `pooled_mean` weights every event equally.

In [12]:
loud_rows = []
for name in ORDER:
    ev = events_by_text[name].dropna(subset=["loudness"])
    c, a = ev[ev.type == "character"]["loudness"], ev[ev.type == "ambient"]["loudness"]
    loud_rows.append(dict(
        text=name, labelled_events=len(ev),
        character_mean=c.mean(), ambient_mean=a.mean(),
        text_loudness_average=(c.mean() + a.mean()) / 2,
        pooled_mean=ev["loudness"].mean(),
        character_std=np.std(c, ddof=0), ambient_std=np.std(a, ddof=0), total_std=np.std(ev["loudness"], ddof=0),
        share_at_3=(ev["loudness"] == 3.0).mean(),
        share_quiet_le2=(ev["loudness"] <= 2).mean(), share_loud_ge4=(ev["loudness"] >= 4).mean(),
        chapter_mean_std=chap[chap.text == name]["loud_mean"].std(ddof=0),
    ))
loudness_df = pd.DataFrame(loud_rows)
loudness_df.round(3)

,text,labelled_events,character_mean,ambient_mean,text_loudness_average,pooled_mean,character_std,ambient_std,total_std,share_at_3,share_quiet_le2,share_loud_ge4,chapter_mean_std
0,What Maisie Knew,1022,3.098,3.085,3.092,3.097,0.435,0.679,0.461,0.796,0.035,0.129,0.078
1,The Golden Bowl,1323,3.005,3.000,3.003,3.005,0.439,0.896,0.475,0.855,0.051,0.074,0.108
2,Swann in Love,596,3.037,3.166,3.101,3.049,0.420,0.608,0.444,0.815,0.044,0.094,0.114


In [13]:
# Do the dictionaries-by-span in the Dickens notebook change the result? (duplicates collapse there)
for name in ORDER:
    ev = events_by_text[name].dropna(subset=["loudness"])
    dedup = ev.drop_duplicates(subset=["type", "span"])
    print(f"{name:18s} labelled events: {len(ev):5d}  unique spans: {len(dedup):5d}  "
          f"std all events: {np.std(ev.loudness):.3f}  std unique spans only: {np.std(dedup.loudness):.3f}")

What Maisie Knew   labelled events:  1022  unique spans:   869  std all events: 0.461  std unique spans only: 0.487
The Golden Bowl    labelled events:  1323  unique spans:   999  std all events: 0.475  std unique spans only: 0.529
Swann in Love      labelled events:   596  unique spans:   547  std all events: 0.444  std unique spans only: 0.451


### Tests
* **Levene** (as in the Dickens notebook): do the three texts differ in the *spread* of loudness?
* **Kruskal–Wallis**: do they differ in the *level* of loudness? (The labels are ordinal and clustered at 3.0, so a rank test is safer than ANOVA.)
* **Welch t-test** per text: are ambient sounds labeled louder or quieter than character sounds?

In [14]:
groups = [events_by_text[n]["loudness"].dropna() for n in ORDER]
print("Levene across texts:        stat = %.3f, p = %.4f" % levene(*groups))
print("Kruskal–Wallis across texts: H = %.3f, p = %.4f" % kruskal(*groups))
for name in ORDER:
    ev = events_by_text[name].dropna(subset=["loudness"])
    t, p = ttest_ind(ev[ev.type == "character"]["loudness"], ev[ev.type == "ambient"]["loudness"], equal_var=False)
    lv = levene(ev[ev.type == "character"]["loudness"], ev[ev.type == "ambient"]["loudness"])
    print(f"{name:18s} character vs ambient: Welch t = {t:6.2f}, p = {p:.4f} | Levene = {lv.statistic:6.2f}, p = {lv.pvalue:.4f}")

Levene across texts:        stat = 1.652, p = 0.1919
Kruskal–Wallis across texts: H = 24.434, p = 0.0000
What Maisie Knew   character vs ambient: Welch t =   0.17, p = 0.8652 | Levene =  30.79, p = 0.0000
The Golden Bowl    character vs ambient: Welch t =   0.05, p = 0.9623 | Levene =  49.28, p = 0.0000
Swann in Love      character vs ambient: Welch t =  -1.58, p = 0.1189 | Levene =  15.82, p = 0.0001


## 4. Who makes the sound? Speech and other sounds
Most `character_sound` events in these novels are verbs of speaking (*said, asked, replied*). Separating them shows how much of each text's soundscape is dialogue and how much is non-verbal sound (laughter, music, bells, steps, voices heard at a distance).

In [15]:
SPEECH = set("""say says said saying ask asks asked asking answer answers answered answering reply replies replied
replying return returned cry cried cries exclaim exclaimed exclaiming declare declared observe observed remark remarked
add added continue continued repeat repeated explain explained tell told telling speak spoke spoken speaking talk talked
talking murmur murmured murmuring whisper whispered whispering call called calling protest protested insist insisted
answer responded respond went inquire inquired enquired demand demanded suggest suggested""".split())


def classify(row):
    words = {w.lower() for w in TOKEN_RE.findall(row["span"])}
    if row["type"] == "ambient":
        return "ambient"
    return "character: speech" if words & SPEECH else "character: other sound"


events["voice"] = events.apply(classify, axis=1)
voice = (events.groupby(["text", "voice"]).size().unstack(fill_value=0))
voice_share = voice.div(voice.sum(axis=1), axis=0).reindex(ORDER)
voice_share.round(3)

voice,ambient,character: other sound,character: speech
text,,,
What Maisie Knew,0.101,0.434,0.465
The Golden Bowl,0.087,0.335,0.579
Swann in Love,0.123,0.426,0.451


## 5. Music
Music words are found in the full text, not only inside sound annotations, because much musical experience in these novels (Swann's *little phrase*, Vinteuil's sonata) is written as perception, memory or feeling rather than as a sound event.

Three layers:
* **strict**: words that refer to music unambiguously (*music, piano, violin, sonata, melody, song, sing, orchestra, opera, Wagner, Vinteuil*, …) and the bigram *little phrase*;
* **contextual**: ambiguous words (*phrase, note, air, play, tune, chord, concert, harmony, theme, strings, instrument*, …) counted as music only within ±25 words of a strict music word;
* **figurative candidates**: musical vocabulary (*harmony, tune, chord, discord, pitch, cadence, concert*, …) **outside** any musical context. These are candidates for musical metaphors (James's "harmony" of a marriage), to be checked in the keyword-in-context table, not proven metaphors.

The lexicons are short and readable on purpose, so they can be discussed and changed in class.

In [16]:
MUSIC_STRICT = set("""music musical musically musician musicians piano pianos pianist pianists pianoforte violin violins
violinist violinists cello violoncello sonata sonatas melody melodies melodious song songs sing sings sang sung singing
singer singers orchestra orchestral orchestras opera operas operatic symphony
symphonies symphonic wagner wagnerian beethoven vinteuil keyboard quartet quartets hymn hymns waltz waltzes composer
composers tremolo andante allegro arpeggio arpeggios lohengrin tristan""".split())
MUSIC_CONTEXT = set("""phrase phrases note notes air airs play plays played playing harmony harmonies harmonious theme themes
string strings bow instrument instruments score hum hummed humming motif motive chant chanted pitch key rhythm cadence
performer performance performed tune tunes chord chords concert concerts""".split())
MUSIC_FIGURATIVE = set("""harmony harmonies harmonious harmoniously discord discords discordant tune tunes chord chords
keynote cadence cadences pitch tune tunes chord chords concert""".split())

music_rows = []
for name in ORDER:
    tok = tokens_by_text[name]
    low = tok["word"].str.lower().to_numpy()
    n = len(low)
    strict = np.isin(low, list(MUSIC_STRICT))
    # "little phrase" counts as strict on the word "phrase"
    lp = np.zeros(n, bool)
    lp[1:] = (low[1:] == "phrase") & (low[:-1] == "little")
    strict = strict | lp
    near = np.zeros(n, bool)
    for i in np.flatnonzero(strict):
        near[max(0, i - MUSIC_WINDOW): i + MUSIC_WINDOW + 1] = True
    contextual = np.isin(low, list(MUSIC_CONTEXT)) & near & ~strict
    figurative = np.isin(low, list(MUSIC_FIGURATIVE)) & ~near & ~strict
    for kind, mask in [("strict", strict), ("contextual", contextual), ("figurative", figurative)]:
        sub = tok.loc[mask]
        for _, r in sub.iterrows():
            music_rows.append(dict(text=name, kind=kind, word=r["word"], i=r["i"], unit=r["unit"],
                                   unit_label=r["unit_label"], rel_pos=r["rel_pos"],
                                   event_id=r["event_id"], char_start=r["char_start"], char_end=r["char_end"]))
music = pd.DataFrame(music_rows)
music["in_sound_event"] = music["event_id"] >= 0


def kwic(name, start, end, width=110):
    s = plain_by_text[name]
    left = " ".join(s[max(0, start - width):start].split())
    right = " ".join(s[end:end + width].split())
    return left, s[start:end], right


music[["left", "kw", "right"]] = music.apply(
    lambda r: pd.Series(kwic(r["text"], r["char_start"], r["char_end"])), axis=1)

# attach the loudness / type of the sound event the music word sits in
ev_key = events.set_index(["text", "event_id"])[["type", "loudness", "span"]]
music = music.merge(ev_key, left_on=["text", "event_id"], right_index=True, how="left")

music_summary = (music.groupby(["text", "kind"]).size().unstack(fill_value=0)
                 .reindex(index=ORDER, columns=["strict", "contextual", "figurative"], fill_value=0))
music_summary["words"] = [len(tokens_by_text[n]) for n in ORDER]
for k in ["strict", "contextual", "figurative"]:
    music_summary[f"{k}_per_10k"] = music_summary[k] / music_summary["words"] * 10000
mus = music[music.kind != "figurative"]
music_summary["music_in_sound_event_share"] = mus.groupby("text")["in_sound_event"].mean().reindex(ORDER)
music_summary.round(2)

kind,strict,contextual,figurative,words,strict_per_10k,contextual_per_10k,figurative_per_10k,music_in_sound_event_share
text,,,,,,,,
What Maisie Knew,8,2,13,98361,0.81,0.20,1.32,0.20
The Golden Bowl,25,9,55,212368,1.18,0.42,2.59,0.38
Swann in Love,244,72,10,91562,26.65,7.86,1.09,0.14


In [17]:
# Which music words carry each text?
(music[music.kind != "figurative"].assign(w=lambda d: d["word"].str.lower())
 .groupby(["text", "w"]).size().rename("n").reset_index()
 .sort_values(["text", "n"], ascending=[True, False]).groupby("text").head(10))

,text,w,n
26,Swann in Love,phrase,43
18,Swann in Love,music,42
28,Swann in Love,pianist,33
38,Swann in Love,sonata,30
29,Swann in Love,piano,21
44,Swann in Love,vinteuil,21
31,Swann in Love,play,16
19,Swann in Love,musical,11
32,Swann in Love,played,11
45,Swann in Love,violin,9


In [18]:
# Sound events that contain music vocabulary, and their loudness
music_event_ids = set(zip(mus.loc[mus.in_sound_event, "text"], mus.loc[mus.in_sound_event, "event_id"]))
events["is_music_event"] = [(t, e) in music_event_ids for t, e in zip(events.text, events.event_id)]
music_loud = (events.dropna(subset=["loudness"]).groupby(["text", "is_music_event"])["loudness"]
              .agg(["size", "mean", lambda s: np.std(s, ddof=0)]).rename(columns={"<lambda_0>": "std"}))
display(music_loud.round(2))
events.loc[events.is_music_event, ["text", "unit_label", "type", "loudness", "span"]]

size  mean   std
text             is_music_event                  
Swann in Love    False            575  3.04  0.43
                 True              21  3.25  0.74
The Golden Bowl  False           1315  3.00  0.47
                 True               8  3.69  0.66
What Maisie Knew False           1020  3.10  0.46
                 True               2  4.00  0.00

,text,unit_label,type,loudness,span
1231,What Maisie Knew,XXVI,ambient,4.0,the music of a man and a woman
1232,What Maisie Knew,XXVI,ambient,4.0,sent up the strum of a guitar and the drawl of...
2221,The Golden Bowl,XI,ambient,NaN,"a low music that, outside one of the windows o..."
2231,The Golden Bowl,XI,ambient,2.0,he so listened to Charlotte’s piano
2242,The Golden Bowl,XII,ambient,NaN,"the big windy hotel, the draughty social hall,..."
2243,The Golden Bowl,XII,ambient,4.0,and resounded with a din in which the wild mus...
2488,The Golden Bowl,XVII,ambient,4.0,but it phrased the large music now in a way th...
2572,The Golden Bowl,XIX,ambient,4.0,the exquisite music
2624,The Golden Bowl,XXII,character,4.0,who played and sang delightfully
2625,The Golden Bowl,XXII,character,4.0,played even “bridge”


## 6. Hearing and listening
The class asks what it means to *have a point of view on sound*. A simple proxy is how often a text names the act of hearing: *hear, heard, listen, listened, overheard, ear(s)*. These are counted in the full text, per chapter and over narrative time.

In [19]:
HEARING = set("hear hears heard hearing listen listens listened listening listener listeners overhear overheard ear ears".split())
tokens["hearing"] = tokens["word"].str.lower().isin(HEARING)
hear_chap = tokens.groupby(["text", "unit"])["hearing"].sum().rename("hearing_words").reset_index()
chap = chap.drop(columns=[c for c in ["hearing_words", "hearing_per_1k"] if c in chap]).merge(hear_chap, on=["text", "unit"], how="left")
chap["hearing_per_1k"] = chap["hearing_words"] / chap["words"] * 1000

mus_chap = mus.groupby(["text", "unit"]).size().rename("music_words").reset_index()
chap = chap.drop(columns=[c for c in ["music_words", "music_per_1k"] if c in chap]).merge(mus_chap, on=["text", "unit"], how="left").fillna({"music_words": 0})
chap["music_per_1k"] = chap["music_words"] / chap["words"] * 1000
chap["text"] = pd.Categorical(chap["text"], ORDER, ordered=True)
chap = chap.sort_values(["text", "unit"]).reset_index(drop=True)

summary["hearing_per_1k"] = [tokens.loc[tokens.text == n, "hearing"].mean() * 1000 for n in ORDER]
summary["music_per_10k"] = (music_summary["strict_per_10k"] + music_summary["contextual_per_10k"]).to_numpy()
summary.round(3)

,text,words,sound_events,character_events,ambient_events,events_per_1k,character_per_1k,ambient_per_1k,SED_pct,mean_event_length_words,labelled_share,chapter_SED_std,chapter_SED_min,chapter_SED_max,hearing_per_1k,music_per_10k
0,What Maisie Knew,98361,1662,1494,168,16.897,15.189,1.708,7.096,4.200,0.615,5.136,5.999,26.443,1.200,1.017
1,The Golden Bowl,212368,2011,1837,174,9.469,8.650,0.819,4.276,4.516,0.658,4.278,2.907,19.226,0.551,1.601
2,Swann in Love,91562,945,829,116,10.321,9.054,1.267,4.138,4.010,0.631,5.112,4.318,23.099,1.649,34.512


## 7. Narrative-time profiles (normalized length)
Every text is cut into 20 bins of 5 % narrative time. In each bin the notebook computes sound events per 1,000 words, music words per 1,000 words, hearing words per 1,000 words, and the mean and standard deviation of loudness.

In [20]:
tokens["bin"] = np.minimum((tokens["rel_pos"] // (100 / N_BINS)).astype(int), N_BINS - 1)
events["bin"] = np.minimum((events["rel_pos"] // (100 / N_BINS)).astype(int), N_BINS - 1)
music["bin"] = np.minimum((music["rel_pos"] // (100 / N_BINS)).astype(int), N_BINS - 1)

bins = tokens.groupby(["text", "bin"]).agg(words=("word", "size"), hearing=("hearing", "sum")).reset_index()
bins = bins.merge(events.groupby(["text", "bin"]).size().rename("events").reset_index(), on=["text", "bin"], how="left")
bins = bins.merge(music[music.kind != "figurative"].groupby(["text", "bin"]).size().rename("music").reset_index(), on=["text", "bin"], how="left")
bins = bins.merge(events.dropna(subset=["loudness"]).groupby(["text", "bin"])["loudness"]
                  .agg(loud_mean="mean", loud_std=lambda s: np.std(s, ddof=0)).reset_index(), on=["text", "bin"], how="left")
bins = bins.fillna({"events": 0, "music": 0})
bins["pct"] = (bins["bin"] + 0.5) * 100 / N_BINS
for c in ["events", "music", "hearing"]:
    bins[f"{c}_per_1k"] = bins[c] / bins["words"] * 1000
bins.round(2).head()

,text,bin,words,hearing,events,music,loud_mean,loud_std,pct,events_per_1k,music_per_1k,hearing_per_1k
0,Swann in Love,0,4579,1,44,17.0,2.90,0.59,2.5,9.61,3.71,0.22
1,Swann in Love,1,4578,17,74,19.0,3.17,0.36,7.5,16.16,4.15,3.71
2,Swann in Love,2,4578,13,73,81.0,3.07,0.25,12.5,15.95,17.69,2.84
3,Swann in Love,3,4578,3,31,16.0,3.00,0.75,17.5,6.77,3.49,0.66
4,Swann in Love,4,4578,4,34,13.0,3.05,0.34,22.5,7.43,2.84,0.87


In [21]:
# Rolling loudness mean ± rolling std over the sequence of labelled sound events
roll = []
for name in ORDER:
    ev = events[(events.text == name)].dropna(subset=["loudness"]).sort_values("start").copy()
    ev["roll_mean"] = ev["loudness"].rolling(ROLLING_EVENTS, center=True, min_periods=ROLLING_EVENTS // 2).mean()
    ev["roll_std"] = ev["loudness"].rolling(ROLLING_EVENTS, center=True, min_periods=ROLLING_EVENTS // 2).std(ddof=0)
    roll.append(ev)
roll = pd.concat(roll)

## 8. Visualizations
Colors: one fixed color per text (blue = *What Maisie Knew*, orange = *The Golden Bowl*, green = *Swann in Love*), violet for character sound, amber for ambient sound, pink for music. Every chart has hover details and is saved as a standalone HTML file.

In [22]:
FONT = "'IBM Plex Sans Condensed', 'Helvetica Neue', Arial, sans-serif"
INK, INK2, GRID, SURFACE = "#1f1e1c", "#5e5d59", "#e7e6e1", "#fcfcfb"
FIGS = {}   # name -> (figure, caption)


def style(fig, title, subtitle=None, height=480, legend=True):
    t = f"<b>{title}</b>"
    if subtitle:
        t += f"<br><span style='font-size:13px;color:{INK2}'>{subtitle}</span>"
    fig.update_layout(
        title=dict(text=t, x=0.01, xanchor="left", y=0.97, yanchor="top", font=dict(size=18)),
        template="plotly_white", font=dict(family=FONT, size=13, color=INK),
        paper_bgcolor=SURFACE, plot_bgcolor=SURFACE, height=height,
        margin=dict(l=70, r=30, t=120 if legend else 95, b=60),
        hoverlabel=dict(bgcolor="white", bordercolor=GRID, font=dict(family=FONT, size=12, color=INK)),
        legend=dict(orientation="h", yanchor="bottom", y=1.01, xanchor="left", x=0, title_text="",
                    font=dict(size=12), traceorder="normal") if legend else None,
        showlegend=legend, bargap=0.18,
    )
    fig.update_xaxes(gridcolor=GRID, linecolor="#c9c8c2", zeroline=False, ticks="outside", tickcolor="#c9c8c2")
    fig.update_yaxes(gridcolor=GRID, linecolor="#c9c8c2", zeroline=False)
    return fig


def save(fig, name, caption=""):
    FIGS[name] = (fig, caption)
    fig.write_html(OUT_DIR / f"{name}.html", include_plotlyjs="directory", full_html=True,
                   config={"displaylogo": False, "responsive": True})
    return fig

### 8.1 Normalized length

In [23]:
fig = go.Figure()
for _, r in length_df.iterrows():
    fig.add_bar(y=[r["text"]], x=[r["words"]], orientation="h", marker_color=TEXT_COLORS[r["text"]],
                name=r["text"], text=[f"{r['words']:,} words · {r['relative_length']:.2f} of the longest"],
                textposition="outside", cliponaxis=False,
                hovertemplate=(f"<b>{r['text']}</b><br>{r['words']:,} words<br>{r['units']} {r['unit_kind']}s, "
                               f"mean {r['mean_unit_words']:,} words<br>relative length {r['relative_length']:.2f}<extra></extra>"))
fig.update_yaxes(categoryorder="array", categoryarray=ORDER[::-1])
fig.update_xaxes(title="words", range=[0, length_df["words"].max() * 1.45])
style(fig, "How long is each text?", "Word counts (tokenizer \\b\\w+\\b, headings excluded). The Golden Bowl is 2.2× What Maisie Knew.", height=340, legend=False)
save(fig, "01_text_length", "Total length. All later measures are rates per 1,000 words or positions in % of narrative time, so the books can be compared despite this gap.")
fig.show()

In [24]:
fig = go.Figure()
for name in ORDER:
    d = chap[chap.text == name]
    fig.add_trace(go.Box(x=d["words"], y=[name] * len(d), name=name, orientation="h",
                         marker=dict(color=TEXT_COLORS[name], size=8, line=dict(color=SURFACE, width=1.5)),
                         line=dict(color=TEXT_COLORS[name], width=1.5), fillcolor="rgba(0,0,0,0)",
                         boxpoints="all", jitter=0.45, pointpos=0, customdata=d[["unit_label", "unit_kind"]],
                         hovertemplate="<b>%{y}</b><br>%{customdata[1]} %{customdata[0]}: %{x:,} words<extra></extra>"))
fig.update_yaxes(categoryorder="array", categoryarray=ORDER[::-1])
fig.update_xaxes(title="words per chapter (Swann in Love: per ~3,000-word segment)")
style(fig, "Chapter lengths", "Each dot is one chapter; the box shows median and quartiles.", height=360, legend=False)
save(fig, "02_chapter_lengths", "Maisie's chapters are short and regular; The Golden Bowl's are longer and much more variable. Swann in Love's segments are cut at ~3,000 words and are regular by construction.")
fig.show()

### 8.2 Sound event density per chapter

In [25]:
fig = make_subplots(rows=3, cols=1, subplot_titles=ORDER, vertical_spacing=0.09)
for row, name in enumerate(ORDER, start=1):
    d = chap[chap.text == name]
    custom = np.stack([d["words"], d["events"], d["SED_pct"], d["loud_mean"].fillna(-1), d["loud_std"].fillna(-1), d["unit_label"]], axis=1)
    for typ in ["character", "ambient"]:
        fig.add_bar(x=d["unit_label"], y=d[f"{typ}_per_1k"], name=f"{typ} sound", marker_color=TYPE_COLORS[typ],
                    marker_line=dict(color=SURFACE, width=1), legendgroup=typ, showlegend=(row == 1),
                    customdata=custom, row=row, col=1,
                    hovertemplate=("<b>%{customdata[5]}</b> · " + typ + ": %{y:.2f} per 1,000 words"
                                   "<br>all events: %{customdata[1]} in %{customdata[0]:,} words"
                                   "<br>SED: %{customdata[2]:.2f} % of words inside sound spans"
                                   "<br>loudness mean %{customdata[3]:.2f}, std %{customdata[4]:.2f}<extra></extra>"))
    mean_rate = summary.loc[summary.text == name, "events_per_1k"].item()
    fig.add_hline(y=mean_rate, line=dict(color=INK2, width=1, dash="dot"), row=row, col=1,
                  annotation_text=f"text mean {mean_rate:.1f}", annotation_position="top left",
                  annotation_font=dict(size=11, color=INK2))
    fig.update_yaxes(title_text="per 1,000 words", row=row, col=1)
fig.update_layout(barmode="stack")
fig.update_xaxes(type="category", tickfont=dict(size=10))
style(fig, "Sound event density per chapter", "Sound events per 1,000 words, stacked by type. Hover for SED (% of words inside sound spans) and loudness.", height=900)
save(fig, "03_sound_density_by_chapter", "Chapter-wise density from the chapter annotations. The dotted line is each text's overall rate.")
fig.show()

In [26]:
fig = go.Figure()
for name in ORDER:
    d = bins[bins.text == name]
    fig.add_scatter(x=d["pct"], y=d["events_per_1k"], mode="lines+markers", name=name,
                    line=dict(color=TEXT_COLORS[name], width=2), marker=dict(size=8, line=dict(color=SURFACE, width=2)),
                    customdata=np.stack([d["words"], d["events"]], axis=1),
                    hovertemplate=f"<b>{name}</b><br>%{{x:.1f}} % of the text<br>%{{y:.2f}} events per 1,000 words<br>%{{customdata[1]}} events in %{{customdata[0]:,}} words<extra></extra>")
fig.update_xaxes(title="narrative time (% of the text)", range=[0, 100], ticksuffix=" %")
fig.update_yaxes(title="sound events per 1,000 words", rangemode="tozero")
style(fig, "Sound density over normalized narrative time", "Each point is 5 % of a text, so books of different lengths share one axis.", height=460)
save(fig, "04_sound_density_narrative_time", "Normalized-length comparison of sound density.")
fig.show()

In [27]:
fig = go.Figure()
cats = ["character: speech", "character: other sound", "ambient"]
cat_colors = {"character: speech": "#4a3aa7", "character: other sound": "#9085e9", "ambient": "#eda100"}
for c in cats:
    fig.add_bar(y=ORDER, x=voice_share[c] * 100, name=c, orientation="h", marker_color=cat_colors[c],
                marker_line=dict(color=SURFACE, width=2),
                customdata=voice.reindex(ORDER)[c],
                hovertemplate="<b>%{y}</b><br>" + c + ": %{x:.1f} % (%{customdata} events)<extra></extra>")
fig.update_layout(barmode="stack")
fig.update_yaxes(categoryorder="array", categoryarray=ORDER[::-1])
fig.update_xaxes(title="share of all sound events", ticksuffix=" %", range=[0, 100])
style(fig, "Whose sound? Speech, other character sounds, ambient sound", "Character sounds whose span contains a verb of speaking (said, asked, replied, …) count as speech.", height=340)
save(fig, "05_speech_vs_sound", "Most annotated sound in all three texts is reported speech; the remainder is what the narrative actually lets us hear.")
fig.show()

### 8.3 Loudness: mean and standard deviation

In [28]:
fig = go.Figure()
for typ in ["character", "ambient"]:
    d = events[(events.type == typ)].dropna(subset=["loudness"])
    fig.add_trace(go.Box(x=d["text"], y=d["loudness"], name=f"{typ} sound", marker_color=TYPE_COLORS[typ],
                         boxmean="sd", line=dict(width=1.5), fillcolor="rgba(0,0,0,0)", boxpoints="outliers",
                         marker=dict(size=6, opacity=0.6), offsetgroup=typ))
fig.update_layout(boxmode="group")
fig.update_xaxes(categoryorder="array", categoryarray=ORDER)
fig.update_yaxes(title="loudness level (1 quiet … 5 loud)", range=[0.5, 5.5], dtick=1)
style(fig, "Loudness distribution with mean ± standard deviation", "Box = quartiles; dashed line and diamond = mean ± 1 standard deviation (population std, as in the Dickens notebook).", height=480)
save(fig, "06_loudness_distribution", "Labelled events only. Most labels sit at 3.0 (the level of speech).")
fig.show()

In [29]:
fig = go.Figure()
measures = [("character_std", "character sound", TYPE_COLORS["character"]),
            ("ambient_std", "ambient sound", TYPE_COLORS["ambient"]),
            ("total_std", "all sound events", "#5e5d59")]
for col, label, color in measures:
    fig.add_bar(x=loudness_df["text"], y=loudness_df[col], name=label, marker_color=color,
                marker_line=dict(color=SURFACE, width=2), text=loudness_df[col].round(2), textposition="outside",
                hovertemplate="<b>%{x}</b><br>" + label + ": std %{y:.3f}<extra></extra>")
fig.update_layout(barmode="group")
fig.update_yaxes(title="standard deviation of loudness", rangemode="tozero")
style(fig, "Loudness standard deviation per text", "How wide is each text's acoustic range? Higher = more movement between quiet and loud.", height=440)
save(fig, "07_loudness_std", "Population standard deviation of the loudness labels.")
fig.show()

In [30]:
fig = make_subplots(rows=3, cols=1, subplot_titles=ORDER, vertical_spacing=0.09, shared_yaxes=True)
for row, name in enumerate(ORDER, start=1):
    d = chap[chap.text == name]
    fig.add_scatter(x=d["unit_label"], y=d["loud_mean"], mode="markers",
                    error_y=dict(type="data", array=d["loud_std"], color=TEXT_COLORS[name], thickness=1.5, width=0),
                    marker=dict(color=TEXT_COLORS[name], size=8, line=dict(color=SURFACE, width=1.5)),
                    name=name, showlegend=False, row=row, col=1,
                    customdata=np.stack([d["loud_std"], d["loud_n"].fillna(0)], axis=1),
                    hovertemplate="<b>%{x}</b><br>mean %{y:.2f} ± %{customdata[0]:.2f}<br>%{customdata[1]:.0f} labelled events<extra></extra>")
    fig.add_hline(y=3, line=dict(color=INK2, width=1, dash="dot"), row=row, col=1)
    fig.update_yaxes(title_text="loudness", range=[1, 5], row=row, col=1)
fig.update_xaxes(type="category", tickfont=dict(size=10))
style(fig, "Loudness per chapter: mean ± standard deviation", "Dot = mean loudness of the chapter's labelled sound events; bar = ±1 std. Dotted line = 3.0 (speech level).", height=880, legend=False)
save(fig, "08_loudness_by_chapter", "Chapters with long error bars move between quiet and loud sounds; chapters without bars contain one labelled event or uniform labels.")
fig.show()

In [31]:
def hex_to_rgba(h, a):
    h = h.lstrip("#")
    return f"rgba({int(h[0:2],16)},{int(h[2:4],16)},{int(h[4:6],16)},{a})"


fig = go.Figure()
for name in ORDER:
    d = roll[(roll.text == name)].dropna(subset=["roll_mean"])
    fig.add_scatter(x=np.r_[d["rel_pos"], d["rel_pos"][::-1]],
                    y=np.r_[d["roll_mean"] + d["roll_std"], (d["roll_mean"] - d["roll_std"])[::-1]],
                    fill="toself", fillcolor=hex_to_rgba(TEXT_COLORS[name], 0.14), line=dict(width=0),
                    hoverinfo="skip", legendgroup=name, showlegend=False)
    fig.add_scatter(x=d["rel_pos"], y=d["roll_mean"], mode="lines", name=name, legendgroup=name,
                    line=dict(color=TEXT_COLORS[name], width=2),
                    customdata=np.stack([d["roll_std"], d["unit_label"], d["span"]], axis=1),
                    hovertemplate=f"<b>{name}</b> · %{{customdata[1]}}<br>%{{x:.1f}} % of the text"
                                  "<br>rolling mean %{y:.2f} ± %{customdata[0]:.2f}<br><i>%{customdata[2]}</i><extra></extra>")
fig.update_xaxes(title="narrative time (% of the text)", range=[0, 100], ticksuffix=" %")
fig.update_yaxes(title="loudness (rolling mean ± rolling std)", range=[1.8, 4.2])
style(fig, f"Loudness over narrative time", f"Rolling mean over windows of {ROLLING_EVENTS} consecutive labelled sound events, shaded ± rolling standard deviation.", height=480)
save(fig, "09_loudness_rolling", "Adapted from the 'loudness average + standard deviation' figure in the Dickens notebook, with a normalized x-axis instead of file order.")
fig.show()

### 8.4 Music

In [32]:
fig = go.Figure()
for kind, label in [("strict", "unambiguous music words"), ("contextual", "ambiguous words in a musical context"),
                    ("figurative", "musical vocabulary outside music (figurative candidates)")]:
    fig.add_bar(x=ORDER, y=music_summary[f"{kind}_per_10k"], name=label, marker_color=MUSIC_COLORS[kind],
                marker_line=dict(color=SURFACE, width=2), offsetgroup="fig" if kind == "figurative" else "lit",
                customdata=music_summary[kind],
                hovertemplate="<b>%{x}</b><br>" + label + ": %{y:.1f} per 10,000 words (%{customdata})<extra></extra>")
fig.update_layout(barmode="stack")
fig.update_yaxes(title="per 10,000 words", rangemode="tozero")
style(fig, "How much music is there?", "Left bar: literal music (stacked). Right bar: musical words used outside any musical context.", height=460)
save(fig, "10_music_rate", "Swann in Love is saturated with music; in James musical vocabulary appears mostly outside any musical scene.")
fig.show()

In [33]:
fig = go.Figure()
for name in ORDER:
    d = bins[bins.text == name]
    fig.add_scatter(x=d["pct"], y=d["music_per_1k"], mode="lines+markers", name=name,
                    line=dict(color=TEXT_COLORS[name], width=2), marker=dict(size=8, line=dict(color=SURFACE, width=2)),
                    customdata=d["music"],
                    hovertemplate=f"<b>{name}</b><br>%{{x:.1f}} % of the text<br>%{{y:.2f}} music words per 1,000 (%{{customdata:.0f}})<extra></extra>")
fig.update_xaxes(title="narrative time (% of the text)", range=[0, 100], ticksuffix=" %")
fig.update_yaxes(title="music words per 1,000 words", rangemode="tozero")
style(fig, "Where does music happen?", "Literal music words (strict + contextual) per 1,000 words, in 5 % slices of each text.", height=460)
save(fig, "11_music_narrative_time", "Peaks mark the musical scenes, e.g. the soirées where Swann hears Vinteuil's sonata.")
fig.show()

In [34]:
cov = (mus.groupby(["text", "in_sound_event"]).size().unstack(fill_value=0).reindex(ORDER).fillna(0))
cov.columns = ["outside sound annotation", "inside a sound event"] if list(cov.columns) == [False, True] else cov.columns
cov_share = cov.div(cov.sum(axis=1), axis=0) * 100
fig = go.Figure()
for c, color in [("inside a sound event", "#d55181"), ("outside sound annotation", "#c9c8c2")]:
    if c in cov_share:
        fig.add_bar(y=ORDER, x=cov_share[c], name=c, orientation="h", marker_color=color,
                    marker_line=dict(color=SURFACE, width=2), customdata=cov[c],
                    hovertemplate="<b>%{y}</b><br>" + c + ": %{x:.1f} % (%{customdata} music words)<extra></extra>")
fig.update_layout(barmode="stack")
fig.update_yaxes(categoryorder="array", categoryarray=ORDER[::-1])
fig.update_xaxes(title="share of literal music words", ticksuffix=" %", range=[0, 100])
style(fig, "Does the sound annotation capture music?", "Share of literal music words that fall inside a <character_sound> or <ambient_sound> span.", height=320)
save(fig, "12_music_annotation_coverage", "Most music in these novels is written as perception, memory or reflection and escapes event-based sound annotation.")
fig.show()

In [35]:
fig = make_subplots(rows=3, cols=1, subplot_titles=ORDER, vertical_spacing=0.09)
for row, name in enumerate(ORDER, start=1):
    d = chap[chap.text == name]
    for col, label, color in [("music_per_1k", "music words", "#d55181"), ("hearing_per_1k", "hearing / listening words", "#1c5cab")]:
        fig.add_bar(x=d["unit_label"], y=d[col], name=label, marker_color=color, legendgroup=label,
                    showlegend=(row == 1), row=row, col=1, marker_line=dict(color=SURFACE, width=1),
                    hovertemplate="<b>%{x}</b><br>" + label + ": %{y:.2f} per 1,000 words<extra></extra>")
    fig.update_yaxes(title_text="per 1,000 words", row=row, col=1)
fig.update_layout(barmode="group")
fig.update_xaxes(type="category", tickfont=dict(size=10))
style(fig, "Music and listening per chapter", "Literal music words next to words of hearing (hear, heard, listen, overheard, ear).", height=880)
save(fig, "13_music_and_hearing_by_chapter", "Where music appears, does the text also dwell on the act of listening?")
fig.show()

In [36]:
fig = go.Figure()
for name in ORDER:
    d = bins[bins.text == name]
    fig.add_scatter(x=d["pct"], y=d["hearing_per_1k"], mode="lines+markers", name=name,
                    line=dict(color=TEXT_COLORS[name], width=2), marker=dict(size=8, line=dict(color=SURFACE, width=2)),
                    hovertemplate=f"<b>{name}</b><br>%{{x:.1f}} % of the text<br>%{{y:.2f}} hearing words per 1,000<extra></extra>")
fig.update_xaxes(title="narrative time (% of the text)", range=[0, 100], ticksuffix=" %")
fig.update_yaxes(title="hearing / listening words per 1,000", rangemode="tozero")
style(fig, "The act of hearing over narrative time", "hear, heard, listen(ed), overheard, ear(s) per 1,000 words, in 5 % slices.", height=440)
save(fig, "14_hearing_narrative_time", "A rough indicator of how often the narration locates sound in a listener.")
fig.show()

### 8.5 Music in context (keyword in context)
The table lists every literal music word and every figurative candidate with its context. It is written into the HTML report with a search box and filters.

In [37]:
kwic_df = music[["text", "unit_label", "rel_pos", "kind", "left", "kw", "right", "in_sound_event", "type", "loudness"]].copy()
kwic_df = kwic_df.sort_values(["text", "rel_pos"], key=lambda s: s.map({n: i for i, n in enumerate(ORDER)}) if s.name == "text" else s)
kwic_df.query("text == 'Swann in Love' and kind == 'strict'").head(15)

,text,unit_label,rel_pos,kind,left,kw,right,in_sound_event,type,loudness
112,Swann in Love,§1,0.043686,strict,that one was indispensable; you must give taci...,pianist,", whom Mme. Verdurin had taken under her patro...",False,NaN,NaN
113,Swann in Love,§1,0.070990,strict,"rin had taken under her patronage that year, a...",Wagner,"as well as that!"" left both Planté and Rubinst...",False,NaN,NaN
114,Swann in Love,§1,0.304712,strict,"a Mme. de Crécy, whom Mme. Verdurin called by ...",pianist,"'s aunt, who looked as though she had, at one ...",False,NaN,NaN
115,Swann in Love,§1,0.435770,strict,er; you had your 'place laid' there. There was...,pianist,"would play, but only if he felt inclined, for ...",False,NaN,NaN
116,Swann in Love,§1,0.473996,strict,"ed to do anything, and, as M. Verdurin used to...",pianist,"suggested playing the Ride of the Valkyries, o...",False,NaN,NaN
117,Swann in Love,§1,0.487102,strict,"here. Liberty Hall, you know!"" If the pianist ...",Tristan,", Mme. Verdurin would protest, not that the mu...",False,NaN,NaN
118,Swann in Love,§1,0.495839,strict,"ested playing the Ride of the Valkyries, or th...",music,"was displeasing to her, but, on the contrary, ...",False,NaN,NaN
119,Swann in Love,§1,0.714270,strict,"big evenings, which were given as seldom as po...",musician,better known. The rest of the time you were qu...,False,NaN,NaN
120,Swann in Love,§1,0.934886,strict,it would make her quite ill to think that the ...,pianist,"'s aunt insisted that he must accompany her, o...",True,character,3.0
121,Swann in Love,§2,3.480702,strict,"depending on Swann for this evening,"" people w...",Opera,"?"" He would secure invitations for her to the ...",False,NaN,NaN


## 9. Export tables and the HTML report

In [38]:
chap.to_csv(OUT_DIR / "chapter_metrics.csv", index=False)
summary.to_csv(OUT_DIR / "text_summary.csv", index=False)
loudness_df.to_csv(OUT_DIR / "loudness_summary.csv", index=False)
length_df.to_csv(OUT_DIR / "length_summary.csv", index=False)
kwic_df.to_csv(OUT_DIR / "music_kwic.csv", index=False)
print("tables written to", OUT_DIR)

tables written to html_visualizations


### The HTML report for the class
One self-contained HTML file (Plotly is embedded, so it works offline in the classroom) with all charts, the key figures, a reading guide for each chart, and a searchable keyword-in-context table of the music passages.

In [39]:
from plotly.offline import get_plotlyjs

esc = html.escape
S = summary.set_index("text"); L = loudness_df.set_index("text"); M = music_summary; LEN = length_df.set_index("text")
mus_lit = music[music.kind != "figurative"]


def top_units(name, col, k=2):
    d = chap[chap.text == name].nlargest(k, col)
    return ", ".join(f"{r.unit_label} ({getattr(r, col):.1f})" for r in d.itertuples())


ml = music_loud.reset_index().rename(columns={"<lambda_0>": "std"})
def ml_val(name, flag, col):
    return ml[(ml.text == name) & (ml.is_music_event == flag)][col].item()


lit_vs_fig = {n: (int(M.loc[n, "strict"] + M.loc[n, "contextual"]), int(M.loc[n, "figurative"])) for n in ORDER}
ratio_music = M.loc["Swann in Love", "strict_per_10k"] + M.loc["Swann in Love", "contextual_per_10k"]
james_music = max(M.loc[n, "strict_per_10k"] + M.loc[n, "contextual_per_10k"] for n in ORDER[:2])
lev = levene(*groups); kw_ = kruskal(*groups)

READING = {
"01_text_length": f"""<p><i>The Golden Bowl</i> has {S.loc['The Golden Bowl','words']:,} words, {LEN.loc['The Golden Bowl','relative_to_Maisie']:.2f} times <i>What Maisie Knew</i> ({S.loc['What Maisie Knew','words']:,}); <i>Swann in Love</i> ({S.loc['Swann in Love','words']:,}) is close to Maisie. Raw counts of sound events would therefore make <i>The Golden Bowl</i> look like the noisiest book. Every comparison below uses rates per 1,000 words or positions in % of the text.</p>""",
"02_chapter_lengths": f"""<p>Maisie has {LEN.loc['What Maisie Knew','units']} short chapters (median {LEN.loc['What Maisie Knew','median_unit_words']:,} words); <i>The Golden Bowl</i> has {LEN.loc['The Golden Bowl','units']} chapters of about {LEN.loc['The Golden Bowl','median_unit_words']:,} words. <i>Swann in Love</i> has no chapters, so it is shown in {LEN.loc['Swann in Love','units']} segments of about 3,000 words cut at paragraph breaks. Its regularity is an artefact of that cut.</p>""",
"03_sound_density_by_chapter": f"""<p>Maisie is the densest text: {S.loc['What Maisie Knew','events_per_1k']:.1f} sound events per 1,000 words, against {S.loc['The Golden Bowl','events_per_1k']:.1f} in <i>The Golden Bowl</i> and {S.loc['Swann in Love','events_per_1k']:.1f} in <i>Swann in Love</i>. A child who learns what she knows by listening to adults talk produces a text full of reported voices. Densest chapters: Maisie {top_units('What Maisie Knew','events_per_1k')}; <i>Golden Bowl</i> {top_units('The Golden Bowl','events_per_1k')}; <i>Swann</i> {top_units('Swann in Love','events_per_1k')}. Ambient sound (amber) is a thin layer everywhere.</p>""",
"04_sound_density_narrative_time": """<p>With narrative time on a common 0–100 % axis, the three books can be read side by side regardless of length. Look for where the curves cross: a peak marks a stretch dominated by dialogue or scenes of listening, a trough a stretch of reflection.</p>""",
"05_speech_vs_sound": f"""<p>Reported speech makes up {voice_share.loc['The Golden Bowl','character: speech']*100:.0f} % of the annotated sound in <i>The Golden Bowl</i>, {voice_share.loc['What Maisie Knew','character: speech']*100:.0f} % in Maisie and {voice_share.loc['Swann in Love','character: speech']*100:.0f} % in <i>Swann in Love</i>. Sound in James is above all the sound of people talking to and about each other. This is the social world mapped through conversation.</p>""",
"06_loudness_distribution": f"""<p>All three texts sit near the speech level of 3 (means {L.loc['What Maisie Knew','pooled_mean']:.2f}, {L.loc['The Golden Bowl','pooled_mean']:.2f}, {L.loc['Swann in Love','pooled_mean']:.2f}); {L['share_at_3'].min()*100:.0f}–{L['share_at_3'].max()*100:.0f} % of all labels are exactly 3.0. The differences in level are small but significant (Kruskal–Wallis H = {kw_.statistic:.1f}, p &lt; 0.001); Maisie is slightly louder, with {L.loc['What Maisie Knew','share_loud_ge4']*100:.0f} % of events at 4 or above.</p>""",
"07_loudness_std": f"""<p>The overall spread is almost the same in the three books (std {L.loc['What Maisie Knew','total_std']:.2f}, {L.loc['The Golden Bowl','total_std']:.2f}, {L.loc['Swann in Love','total_std']:.2f}; Levene p = {lev.pvalue:.2f}). Within each book the ambient sounds vary far more than the character sounds (Levene p &lt; 0.001 in all three): the acoustic range comes from the world around the characters, not from their voices. <i>The Golden Bowl</i> has the widest ambient range ({L.loc['The Golden Bowl','ambient_std']:.2f}).</p>""",
"08_loudness_by_chapter": f"""<p>Chapters with the widest loudness range: Maisie {top_units('What Maisie Knew','loud_std')}; <i>Golden Bowl</i> {top_units('The Golden Bowl','loud_std')}; <i>Swann</i> {top_units('Swann in Love','loud_std')}. Check the number of labelled events in the tooltip: a chapter with five labels can show a large spread by chance.</p>""",
"09_loudness_rolling": f"""<p>The rolling mean stays in a narrow band around 3 in all three texts. The shaded band shows where the soundscape opens up to quiet or loud sounds. The tooltip shows the sound event at that point.</p>""",
"10_music_rate": f"""<p><i>Swann in Love</i> has {ratio_music:.1f} literal music words per 10,000 words, about {ratio_music/james_music:.0f} times more than either James novel. In James, musical vocabulary used outside any musical scene outnumbers literal music: <i>The Golden Bowl</i> has {lit_vs_fig['The Golden Bowl'][0]} literal and {lit_vs_fig['The Golden Bowl'][1]} figurative candidates (<i>harmony</i>, <i>pitch</i>, <i>tune</i>), Maisie {lit_vs_fig['What Maisie Knew'][0]} and {lit_vs_fig['What Maisie Knew'][1]}. James borrows music to describe relations; Proust describes listening to music.</p>""",
"11_music_narrative_time": """<p><i>Swann in Love</i> has two musical centres: the Verdurin evenings around 10–25 %, where Swann first identifies Vinteuil's sonata and its <i>little phrase</i>, and the soirée at Mme de Saint-Euverte's around 70–85 %, where he hears the phrase again after losing Odette. The James novels stay close to zero, with small peaks at evening parties.</p>""",
"12_music_annotation_coverage": f"""<p>Only {M.loc['Swann in Love','music_in_sound_event_share']*100:.0f} % of the music words in <i>Swann in Love</i> fall inside an annotated sound event ({M.loc['The Golden Bowl','music_in_sound_event_share']*100:.0f} % in <i>The Golden Bowl</i>, {M.loc['What Maisie Knew','music_in_sound_event_share']*100:.0f} % in Maisie). Proust writes music as perception, memory and thought, which event-based annotation does not see. When music is annotated, it is labelled louder and more variably than other sounds: in <i>Swann</i> mean {ml_val('Swann in Love', True, 'mean'):.2f} (std {ml_val('Swann in Love', True, 'std'):.2f}) against {ml_val('Swann in Love', False, 'mean'):.2f} ({ml_val('Swann in Love', False, 'std'):.2f}).</p>""",
"13_music_and_hearing_by_chapter": f"""<p>Words of hearing are most frequent in <i>Swann in Love</i> ({S.loc['Swann in Love','hearing_per_1k']:.2f} per 1,000 words), then Maisie ({S.loc['What Maisie Knew','hearing_per_1k']:.2f}), then <i>The Golden Bowl</i> ({S.loc['The Golden Bowl','hearing_per_1k']:.2f}). In Swann the peaks of hearing ({top_units('Swann in Love','hearing_per_1k')}) sit next to the first musical peak: the novel stops on the listener as well as on the music.</p>""",
"14_hearing_narrative_time": """<p>A rough measure of how often sound is located in a listener (<i>hear, heard, listen, overheard, ear</i>). In Maisie, overhearing is the child's way of knowing; compare where her curve rises with the density peaks in the chart above.</p>""",
}

SECTIONS = [
    ("length", "Length and structure", ["01_text_length", "02_chapter_lengths"]),
    ("density", "Sound event density", ["03_sound_density_by_chapter", "04_sound_density_narrative_time", "05_speech_vs_sound"]),
    ("loudness", "Loudness and its standard deviation", ["06_loudness_distribution", "07_loudness_std", "08_loudness_by_chapter", "09_loudness_rolling"]),
    ("music", "Music", ["10_music_rate", "11_music_narrative_time", "12_music_annotation_coverage", "13_music_and_hearing_by_chapter"]),
    ("hearing", "Point of view: the act of hearing", ["14_hearing_narrative_time"]),
]

# key-figure table
rows_html = ""
for n in ORDER:
    rows_html += (f"<tr><th scope='row'><span class='swatch' style='background:{TEXT_COLORS[n]}'></span><i>{esc(n)}</i>"
                  f"<span class='sub'>{TEXTS[n]['author']}, {TEXTS[n]['year']}</span></th>"
                  f"<td>{S.loc[n,'words']:,}</td><td>{LEN.loc[n,'relative_length']:.2f}</td>"
                  f"<td>{LEN.loc[n,'units']} {'ch.' if LEN.loc[n,'unit_kind']=='chapter' else 'seg.'}</td>"
                  f"<td>{S.loc[n,'events_per_1k']:.1f}</td><td>{S.loc[n,'SED_pct']:.1f} %</td>"
                  f"<td>{L.loc[n,'pooled_mean']:.2f}</td><td>{L.loc[n,'character_std']:.2f}</td><td>{L.loc[n,'ambient_std']:.2f}</td><td>{L.loc[n,'total_std']:.2f}</td>"
                  f"<td>{S.loc[n,'music_per_10k']:.1f}</td><td>{M.loc[n,'music_in_sound_event_share']*100:.0f} %</td>"
                  f"<td>{S.loc[n,'hearing_per_1k']:.2f}</td></tr>")

key_table = f"""<div class="table-wrap"><table class="keys">
<thead><tr><th scope="col">Text</th><th scope="col">Words</th><th scope="col">Relative length</th><th scope="col">Units</th>
<th scope="col">Sound events / 1,000 words</th><th scope="col">SED (% of words in sound)</th><th scope="col">Loudness mean</th>
<th scope="col">Std character</th><th scope="col">Std ambient</th><th scope="col">Std all</th>
<th scope="col">Music words / 10,000</th><th scope="col">Music inside sound events</th><th scope="col">Hearing words / 1,000</th></tr></thead>
<tbody>{rows_html}</tbody></table></div>"""

nav = "".join(f"<a href='#{sid}'>{esc(t)}</a>" for sid, t, _ in SECTIONS) + "<a href='#kwic'>Music in context</a><a href='#method'>Method</a>"

sections_html = ""
for sid, title, names in SECTIONS:
    sections_html += f"<section id='{sid}'><h2>{esc(title)}</h2>"
    for name in names:
        fig, cap = FIGS[name]
        div = fig.to_html(full_html=False, include_plotlyjs=False, config={"displaylogo": False, "responsive": True}, div_id=f"fig-{name}")
        sections_html += (f"<figure class='plate'>{div}<figcaption>{esc(cap)}</figcaption></figure>"
                          f"<div class='reading'><h3>Reading the chart</h3>{READING.get(name,'')}</div>")
    sections_html += "</section>"

kw = kwic_df.copy()
kw_records = [dict(t=r.text, u=str(r.unit_label), p=round(float(r.rel_pos), 1), k=r.kind, l=r.left, w=r.kw, r=r.right,
                   s=bool(r.in_sound_event), ld=(None if pd.isna(r.loudness) else float(r.loudness))) for r in kw.itertuples()]
kwic_json = json.dumps(kw_records, ensure_ascii=False).replace("</", "<\\/")

method_html = f"""
<ul>
<li><b>Data.</b> Sound-annotated TEI files with <code>&lt;character_sound&gt;</code> and <code>&lt;ambient_sound&gt;</code> spans and dictionary-based loudness labels (1–5). {int(events.loudness.notna().sum()):,} of {len(events):,} sound events ({events.loudness.notna().mean()*100:.0f} %) carry a label; unlabelled events count for density but not for loudness.</li>
<li><b>Words.</b> Tokenizer <code>\\b\\w+\\b</code> as in the Dickens notebook; chapter numerals, book and part headings and the title line are excluded.</li>
<li><b>Chapters.</b> From the <code>&lt;chapter counter&gt;</code> milestones. Maisie's unnumbered opening section is shown as “Prologue”. <i>Swann in Love</i> has no milestones and is cut into segments of about {PROUST_SEGMENT_WORDS:,} words at paragraph breaks.</li>
<li><b>SED.</b> Events per 1,000 words, and the share of words inside sound spans (the measure behind <code>SED_without_nan</code> in the Dickens notebook).</li>
<li><b>Standard deviation.</b> Population standard deviation (<code>np.std</code>, ddof = 0) of the loudness labels, per text, sound type and chapter; the rolling version uses windows of {ROLLING_EVENTS} labelled events. Every occurrence of a span counts (the Dickens notebook stored spans as dictionary keys, which merges repeated spans like “she said”).</li>
<li><b>Music.</b> Unambiguous music words, plus ambiguous words (<i>phrase, note, air, play, tune, concert, harmony</i> …) within ±{MUSIC_WINDOW} words of an unambiguous one. Musical words outside any musical context are listed as figurative candidates; they are candidates, to be read in context.</li>
<li><b>Annotation repair.</b> In <i>What Maisie Knew</i> a stray opening tag (chapter XX) and a stray closing tag (chapter XXVI) cancelled out, so that the file stayed valid XML while one “sound event” covered 26,000 words. Both are corrected in the notebook before parsing.</li>
</ul>"""

page = f"""<!doctype html>
<html lang="en"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width, initial-scale=1, viewport-fit=cover">
<title>Sound in James and Proust</title>
<link rel="preconnect" href="https://fonts.googleapis.com"><link rel="preconnect" href="https://fonts.gstatic.com" crossorigin>
<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Spectral:ital,wght@0,400;0,600;1,400&family=IBM+Plex+Sans+Condensed:wght@400;500;600&display=swap">
<style>
/* Layout: a single reading column for prose, chart plates allowed to run wider; sticky section index on top. */
:root {{
  --paper:#f7f7f4; --ink:#1f1e1c; --ink-2:#5e5d59; --rule:#dcdad3; --accent:#4a3aa7; --plate:#fcfcfb; --tint:#efeee9;
  --serif:'Spectral', Georgia, 'Times New Roman', serif; --sans:'IBM Plex Sans Condensed', 'Helvetica Neue', Arial, sans-serif;
}}
@media (prefers-color-scheme: dark) {{ :root:not([data-theme="light"]) {{ --paper:#171716; --ink:#ecebe6; --ink-2:#b4b2a9; --rule:#3a3936; --accent:#a99ff0; --tint:#24231f; color-scheme:dark; }} }}
:root[data-theme="dark"] {{ --paper:#171716; --ink:#ecebe6; --ink-2:#b4b2a9; --rule:#3a3936; --accent:#a99ff0; --tint:#24231f; color-scheme:dark; }}
* {{ box-sizing:border-box; }}
body {{ margin:0; background:var(--paper); color:var(--ink); font-family:var(--serif); font-size:1.075rem; line-height:1.6; }}
.wrap {{ max-width:1180px; margin:0 auto; padding-inline:clamp(16px,4vw,40px); padding-block:0 4rem; }}
header.top {{ padding-block:3rem 1.5rem; border-bottom:1px solid var(--rule); }}
.eyebrow {{ font-family:var(--sans); font-size:.8rem; letter-spacing:.08em; text-transform:uppercase; color:var(--ink-2); }}
h1 {{ font-weight:600; font-size:clamp(2rem,5vw,3.2rem); line-height:1.1; margin:.4rem 0 1rem; text-wrap:balance; }}
h1 i {{ font-weight:400; }}
h2 {{ font-weight:600; font-size:1.7rem; margin:3.5rem 0 1rem; padding-top:1rem; border-top:1px solid var(--rule); text-wrap:balance; }}
h3 {{ font-family:var(--sans); font-size:.78rem; letter-spacing:.08em; text-transform:uppercase; color:var(--ink-2); margin:0 0 .3rem; font-weight:600; }}
p, li {{ max-width:68ch; }}
.lede {{ font-size:1.2rem; color:var(--ink); max-width:62ch; }}
nav.index {{ position:sticky; top:env(safe-area-inset-top,0px); z-index:5; background:var(--paper); border-bottom:1px solid var(--rule);
  display:flex; gap:.4rem 1.3rem; flex-wrap:wrap; padding-block:.6rem; font-family:var(--sans); font-size:.9rem; }}
nav.index a {{ color:var(--ink-2); text-decoration:none; }}
nav.index a:hover, nav.index a:focus-visible {{ color:var(--accent); text-decoration:underline; }}
.legend-texts {{ display:flex; flex-wrap:wrap; gap:.5rem 1.4rem; font-family:var(--sans); font-size:.95rem; margin:1rem 0 0; }}
.swatch {{ display:inline-block; width:.8rem; height:.8rem; border-radius:2px; margin-right:.45rem; vertical-align:-.05rem; }}
.table-wrap {{ overflow-x:auto; margin:1.5rem 0; border:1px solid var(--rule); border-radius:6px; }}
table {{ border-collapse:collapse; width:100%; font-family:var(--sans); font-size:.9rem; font-variant-numeric:tabular-nums; }}
th, td {{ padding:.55rem .7rem; text-align:right; border-bottom:1px solid var(--rule); vertical-align:top; }}
thead th {{ font-weight:500; color:var(--ink-2); font-size:.8rem; line-height:1.25; background:var(--tint); }}
th[scope=row], thead th:first-child {{ text-align:left; white-space:nowrap; }}
th[scope=row] {{ font-family:var(--serif); font-weight:400; font-size:1rem; }}
.sub {{ display:block; font-family:var(--sans); font-size:.78rem; color:var(--ink-2); margin-left:1.25rem; }}
tbody tr:last-child th, tbody tr:last-child td {{ border-bottom:0; }}
figure.plate {{ margin:1.5rem 0 .6rem; background:var(--plate); border:1px solid var(--rule); border-radius:6px; padding:.5rem; overflow:hidden; }}
figure.plate figcaption {{ font-family:var(--sans); font-size:.85rem; color:#5e5d59; padding:.3rem .6rem .4rem; }}
.reading {{ margin:0 0 2.2rem; padding-left:1rem; border-left:2px solid var(--accent); }}
.reading p {{ margin:.2rem 0; }}
.controls {{ display:flex; flex-wrap:wrap; gap:.8rem 1.2rem; align-items:end; font-family:var(--sans); font-size:.9rem; margin:1rem 0; }}
.controls label {{ display:flex; flex-direction:column; gap:.25rem; color:var(--ink-2); }}
.controls select, .controls input {{ font:inherit; color:var(--ink); background:var(--paper); border:1px solid var(--rule); border-radius:4px; padding:.4rem .5rem; min-width:0; }}
.controls input {{ width:min(22rem, 80vw); }}
.count {{ color:var(--ink-2); }}
.kwic-wrap {{ max-height:36rem; overflow:auto; border:1px solid var(--rule); border-radius:6px; }}
table.kwic td {{ text-align:left; font-family:var(--serif); font-size:.95rem; }}
table.kwic td.l {{ text-align:right; color:var(--ink-2); width:42%; }}
table.kwic td.r {{ color:var(--ink-2); width:42%; }}
table.kwic td.k {{ font-weight:600; white-space:nowrap; color:var(--accent); }}
table.kwic td.meta {{ font-family:var(--sans); font-size:.78rem; color:var(--ink-2); white-space:nowrap; }}
table.kwic thead th {{ position:sticky; top:0; text-align:left; }}
.pill {{ display:inline-block; font-family:var(--sans); font-size:.72rem; padding:.05rem .4rem; border-radius:999px; border:1px solid var(--rule); color:var(--ink-2); }}
.pill.s {{ border-color:var(--accent); color:var(--accent); }}
code {{ font-family:var(--sans); font-size:.9em; background:var(--tint); padding:.05rem .3rem; border-radius:3px; }}
footer {{ margin-top:3rem; font-family:var(--sans); font-size:.85rem; color:var(--ink-2); }}
@media (max-width:640px) {{ table.kwic td.l, table.kwic td.r {{ min-width:12rem; }} }}
</style>
<script>{get_plotlyjs()}</script>
</head><body><div class="wrap">
<header class="top">
<div class="eyebrow">Sound in Novels · Point of view and the soundscape</div>
<h1>Sound in James and Proust: <i>What Maisie Knew</i>, <i>The Golden Bowl</i>, <i>Swann in Love</i></h1>
<p class="lede">How much sound do these novels contain, how loud is it, and where is the music? Every measure is normalized by length, so a 92,000-word segment of <i>In Search of Lost Time</i> can be read next to a 212,000-word late James novel.</p>
<div class="legend-texts">{''.join(f"<span><span class='swatch' style='background:{TEXT_COLORS[n]}'></span><i>{esc(n)}</i></span>" for n in ORDER)}
<span><span class='swatch' style='background:{TYPE_COLORS["character"]}'></span>character sound</span><span><span class='swatch' style='background:{TYPE_COLORS["ambient"]}'></span>ambient sound</span><span><span class='swatch' style='background:#d55181'></span>music</span></div>
</header>
<nav class="index" aria-label="Sections">{nav}</nav>
<section id="keys"><h2>Key figures</h2>{key_table}
<p>Loudness runs from 1 (very quiet) to 5 (very loud); 3 is the level of ordinary speech. “Std” is the standard deviation of the loudness labels: how far the sounds of a text move away from their average level.</p></section>
{sections_html}
<section id="kwic"><h2>Music in context</h2>
<p>Every music word found in the three texts, with its context. “Figurative” rows are musical words outside any musical scene; read them to decide which are metaphors. A filled tag marks words that fall inside an annotated sound event.</p>
<div class="controls">
<label for="f-text">Text<select id="f-text"><option value="">All texts</option>{''.join(f'<option>{esc(n)}</option>' for n in ORDER)}</select></label>
<label for="f-kind">Layer<select id="f-kind"><option value="">All layers</option><option value="strict">Unambiguous</option><option value="contextual">Contextual</option><option value="figurative">Figurative candidates</option></select></label>
<label for="f-q">Search<input id="f-q" type="search" placeholder="e.g. phrase, Vinteuil, harmony"></label>
<span class="count" id="f-count"></span></div>
<div class="kwic-wrap"><table class="kwic"><thead><tr><th>Where</th><th class="l">Left context</th><th>Word</th><th>Right context</th></tr></thead><tbody id="kwic-body"></tbody></table></div>
</section>
<section id="method"><h2>Method</h2>{method_html}</section>
<footer>Generated by <code>Sound_in_James_and_Proust.ipynb</code>. Charts are also saved one by one in <code>html_visualizations/</code>.</footer>
</div>
<script>
const KWIC = {kwic_json};
const LABEL = {{strict:"unambiguous", contextual:"contextual", figurative:"figurative"}};
const body = document.getElementById("kwic-body"), ft = document.getElementById("f-text"), fk = document.getElementById("f-kind"), fq = document.getElementById("f-q"), fc = document.getElementById("f-count");
function esc(s){{ return String(s).replace(/[&<>"]/g, c => ({{"&":"&amp;","<":"&lt;",">":"&gt;",'"':"&quot;"}})[c]); }}
function render(){{
  const t = ft.value, k = fk.value, q = fq.value.trim().toLowerCase();
  const rows = KWIC.filter(r => (!t || r.t === t) && (!k || r.k === k) && (!q || (r.l + " " + r.w + " " + r.r).toLowerCase().includes(q)));
  fc.textContent = rows.length.toLocaleString("en") + " of " + KWIC.length.toLocaleString("en") + " passages";
  body.innerHTML = rows.slice(0, 600).map(r => `<tr><td class="meta"><i>${{esc(r.t)}}</i><br>${{esc(r.u)}} · ${{r.p}} %<br><span class="pill${{r.s ? " s" : ""}}">${{LABEL[r.k]}}${{r.s ? " · sound event" + (r.ld !== null ? " " + r.ld.toFixed(1) : "") : ""}}</span></td><td class="l">${{esc(r.l)}}</td><td class="k">${{esc(r.w)}}</td><td class="r">${{esc(r.r)}}</td></tr>`).join("");
}}
[ft, fk].forEach(e => e.addEventListener("change", render)); fq.addEventListener("input", render);
ft.value = "Swann in Love"; fk.value = "strict"; render();
</script>
</body></html>"""

REPORT_PATH.write_text(page, encoding="utf-8")
print(f"report written to {REPORT_PATH} ({REPORT_PATH.stat().st_size/1e6:.1f} MB)")

report written to Sound_in_James_and_Proust_report.html (5.5 MB)


## 10. Try it yourself
The cells below are a playground. Each one asks a small question of the three novels. Change the words in quotation marks and run the cell again (**Shift + Enter**). Nothing you do here can damage the data.

**Tools**
* `concordance("word")`: every passage where a word or phrase occurs, with its context (a keyword-in-context table, as in a printed concordance).
* `word_table(["word", "word"])`: how often words occur in each novel, in absolute numbers and per 10,000 words.
* `compare_words(["word", "word"])`: a chart of where in each novel these words occur, over normalized narrative time.
* `sound_events(text, contains="word", min_loudness=4)`: the annotated sound events, filtered.

In [40]:
def concordance(phrase, text=None, width=90, max_rows=40):
    """Keyword in context for a word or phrase (case-insensitive, whole words)."""
    pat = re.compile(r"\b" + r"\s+".join(map(re.escape, phrase.split())) + r"\b", re.I)
    rows = []
    for name in ([text] if text else ORDER):
        s = plain_by_text[name]
        for m in pat.finditer(s):
            rows.append(dict(text=name, position_pct=round(m.start() / len(s) * 100, 1),
                             left=" ".join(s[max(0, m.start() - width):m.start()].split()),
                             match=m.group(), right=" ".join(s[m.end():m.end() + width].split())))
    out = pd.DataFrame(rows, columns=["text", "position_pct", "left", "match", "right"])
    print(f'"{phrase}": {len(out)} occurrences' + (f" (showing the first {max_rows})" if len(out) > max_rows else ""))
    return out.head(max_rows).style.set_properties(subset=["left"], **{"text-align": "right"}).hide(axis="index")


def word_table(words):
    """Counts of each word per novel, absolute and per 10,000 words."""
    words = [w.lower() for w in words]
    rows = []
    for name in ORDER:
        low = tokens_by_text[name]["word"].str.lower()
        for w in words:
            n = int((low == w).sum())
            rows.append(dict(text=name, word=w, count=n, per_10k=round(n / len(low) * 10000, 2)))
    return pd.DataFrame(rows).pivot(index="word", columns="text", values=["count", "per_10k"]).reindex(columns=ORDER, level=1)


def compare_words(words, label=None):
    """Where do these words occur? Occurrences per 1,000 words in 5 % slices of each novel."""
    words = {w.lower() for w in words}
    label = label or ", ".join(sorted(words))
    fig = go.Figure()
    for name in ORDER:
        tok = tokens[tokens.text == name]
        hit = tok["word"].str.lower().isin(words)
        g = pd.DataFrame({"bin": tok["bin"], "hit": hit}).groupby("bin").agg(words=("hit", "size"), hits=("hit", "sum"))
        fig.add_scatter(x=(g.index + 0.5) * 100 / N_BINS, y=g["hits"] / g["words"] * 1000, mode="lines+markers",
                        name=name, line=dict(color=TEXT_COLORS[name], width=2), marker=dict(size=7),
                        customdata=g["hits"], hovertemplate=f"<b>{name}</b><br>%{{x:.1f}} %<br>%{{y:.2f}} per 1,000 words (%{{customdata}})<extra></extra>")
    fig.update_xaxes(title="narrative time (% of the text)", range=[0, 100], ticksuffix=" %")
    fig.update_yaxes(title="per 1,000 words", rangemode="tozero")
    return style(fig, f"Where do these words occur? {label}", "Occurrences per 1,000 words in 5 % slices of each novel.", height=420)


def sound_events(text=None, contains=None, min_loudness=None, max_loudness=None, kind=None, max_rows=40):
    """Browse the annotated sound events. kind = 'character' or 'ambient'."""
    d = events.copy()
    if text: d = d[d.text == text]
    if kind: d = d[d.type == kind]
    if contains: d = d[d.span.str.contains(contains, case=False, regex=False)]
    if min_loudness is not None: d = d[d.loudness >= min_loudness]
    if max_loudness is not None: d = d[d.loudness <= max_loudness]
    print(f"{len(d)} sound events" + (f" (showing the first {max_rows})" if len(d) > max_rows else ""))
    return d[["text", "unit_label", "type", "loudness", "span"]].rename(columns={"unit_label": "chapter"}).head(max_rows)

In [41]:
# Swann's "little phrase": every occurrence, with context
concordance("little phrase")

"little phrase": 29 occurrences


text,position_pct,left,match,right
Swann in Love,11.900000,"iends, but, vividly as he could recall the exquisite and inexpressible pleasure which the",little phrase,"had given him, and could see, still, before his eyes the forms that it had traced in outl"
Swann in Love,12.300000,"ngeade; it's well earned!"" Swann began to tell Odette how he had fallen in love with that",little phrase,". When their hostess, who was a little way off, called out, ""Well! It looks to me as thoug"
Swann in Love,12.400000,"he had done, and at what period in his life he had composed the sonata;--what meaning the",little phrase,"could have had for him, that was what Swann wanted most to know. But none of these peopl"
Swann in Love,15.200000,"f Odette, the pianist would play to them--for their two selves, and for no one else--that",little phrase,"by Vinteuil which was, so to speak, the national anthem of their love. He began, always,"
Swann in Love,15.300000,"mote, in colour quite different, velvety with the radiance of some intervening light, the",little phrase,"appeared, dancing, pastoral, interpolated, episodic, belonging to another world. It passe"
Swann in Love,15.800000,"o long outside, with his little girl, before going to the Verdurins' that, as soon as the",little phrase,"had been rendered by the pianist, Swann would discover that it was almost time for Odette"
Swann in Love,25.100000,"er will, was to be played at her funeral); but he would ask her, instead, to give him the",little phrase,"from Vinteuil's sonata. It was true that Odette played vilely, but often the fairest impr"
Swann in Love,25.100000,sen out of a jumble of wrong notes struck by unskilful fingers upon a tuneless piano. The,little phrase,"was associated still, in Swann's mind, with his love for Odette. He felt clearly that thi"
Swann in Love,25.200000,fice so many of his intellectual and social interests to this imaginary pleasure. But the,little phrase,", as soon as it struck his ear, had the power to liberate in him the room that was needed"
Swann in Love,25.300000,"reality superior to that of other concrete things. This thirst for an untasted charm, the",little phrase,"would stimulate it anew in him, but without bringing him any definite gratification to as"


In [42]:
# Words for listening and for silence, side by side
word_table(["heard", "listened", "overheard", "silence", "silent", "noise"])

count                                        per_10k                              
text      What Maisie Knew The Golden Bowl Swann in Love What Maisie Knew The Golden Bowl Swann in Love
word                                                                                                   
heard                 55.0            33.0          62.0             5.59            1.55          6.77
listened               3.0            17.0           8.0             0.30            0.80          0.87
noise                  3.0             3.0           1.0             0.30            0.14          0.11
overheard              0.0             1.0           0.0             0.00            0.05          0.00
silence               34.0            52.0           6.0             3.46            2.45          0.66
silent                13.0            17.0           4.0             1.32            0.80          0.44

In [43]:
# Where does silence fall in each novel?
compare_words(["silence", "silent", "silently", "hush", "hushed", "still", "stillness"], label="silence").show()

In [44]:
# Which sounds are labelled loudest (4 or more) in The Golden Bowl?
sound_events("The Golden Bowl", min_loudness=4)

98 sound events (showing the first 40)


,text,chapter,type,loudness,span
1669,The Golden Bowl,I,character,4.0,the girl had laughed at him
1673,The Golden Bowl,I,character,4.0,she laughed
1691,The Golden Bowl,I,character,4.0,he had laughed for this
1705,The Golden Bowl,II,character,4.0,The way she laughed at him
1736,The Golden Bowl,II,character,4.0,laugh
1739,The Golden Bowl,II,character,4.0,as she called
1759,The Golden Bowl,II,character,4.0,his companion laugh
1775,The Golden Bowl,II,character,4.0,he rang out clear
1779,The Golden Bowl,II,character,4.0,His friend gave a laugh
1786,The Golden Bowl,III,character,4.0,Mrs. Assingham took it up with an irony beyond...


In [45]:
# Every annotated sound event that mentions the violin
sound_events(contains="violin")

7 sound events


,text,chapter,type,loudness,span
3793,Swann in Love,§4,ambient,3.0,he had heard a piece of music played on the pi...
4099,Swann in Love,§11,ambient,NaN,heralded by the waving tremolo of the violin-part
4443,Swann in Love,§23,ambient,NaN,the violin had risen to a series of high notes
4452,Swann in Love,§23,ambient,4.0,There are in the music of the violin
4461,Swann in Love,§24,character,2.0,Swann now heard between piano and violin
4464,Swann in Love,§24,ambient,3.0,the violin heard and answered it
4468,Swann in Love,§24,ambient,NaN,violinist


This jupyter notebook has been prepared with Claude Opus 5.5 Medium.
Last update on 2026/10/06.